In [26]:
import os
import sys

print("=" * 60)
print("SETUP")
print("=" * 60)

# Detect environment
on_colab = 'google.colab' in sys.modules
print(f"Environment: {'Colab' if on_colab else 'Local'}")

if on_colab:
    base_dir = "/content"
    os.chdir(base_dir)

    # Clone repo if missing
    if not os.path.exists('trading-signals'):
        print("Cloning repo...")
        os.system('git clone https://github.com/tom-curtis/trading-signals.git')

    # Enter project directory
    os.chdir('trading-signals')

    print("Pulling latest...")
    os.system('git pull origin main -q')

    data_path = '/content/data'
else:
    os.chdir('.')  # local
    data_path = 'data'

project_dir = os.getcwd()
print(f"Project dir: {project_dir}")

# Ensure data dir exists
os.makedirs(data_path, exist_ok=True)

# Download Kaggle data if missing
csv_path = f'{data_path}/Combined_News_DJIA.csv'

if not os.path.exists(csv_path):
    print(f"Downloading Kaggle data to {data_path}...")
    os.system('pip install kaggle -q')
    os.system(f'kaggle datasets download -d aaron7sun/stocknews -p {data_path} --unzip -q')

# Add project to path
sys.path.insert(0, project_dir)

# Validate setup
print(f"✓ Data: {os.path.exists(csv_path)}")
print(f"✓ src.data_loader: ", end="")
try:
    from src.data_loader import load_kaggle_data
    print("OK")
except Exception as e:
    print(f"FAIL ({e})")

print("=" * 60)
print("Setup complete!\n")

SETUP
Environment: Colab
Pulling latest...
Project dir: /content/trading-signals
✓ Data: True
✓ src.data_loader: FAIL (cannot import name 'load_kaggle_data' from 'src.data_loader' (/content/trading-signals/src/data_loader.py))
Setup complete!



# Predicting Market Direction from News Headlines

This project investigates whether daily news headlines contain predictive signal for stock market movements. The task is formulated as a binary classification problem, where the objective is to predict whether the market will move up or down on the following trading day.

Two data sources are used: a dataset of news headlines and historical DJIA price data. These are combined to construct a dataset where each observation corresponds to a single trading day.

The approach consists of three components. First, headline text is used to train a sequence model for direction prediction. Second, price data is used to identify anomalous market conditions. Finally, these signals are combined to evaluate whether anomaly awareness improves decision-making.

## Data Preparation

The dataset consists of two sources: daily news headlines and historical DJIA market data. These are processed separately before being combined into a single modelling dataset.

In [27]:
from src.data_loader import (
    load_headlines_csv,
    load_prices_csv,
    aggregate_headlines_by_day,
    add_price_features,
    merge_market_and_headlines,
    add_next_day_target,
    split_by_ratio
)

import pandas as pd

Both datasets are loaded and dates are parsed into a consistent datetime format. Rows with missing or invalid values are removed. Headline text is cleaned by removing empty entries and normalising whitespace.

In [28]:
headlines_df = load_headlines_csv(f"{data_path}/RedditNews.csv")
prices_df = load_prices_csv(f"{data_path}/upload_DJIA_table.csv")

print(headlines_df.shape)
print(prices_df.shape)

headlines_df.head()

(73607, 2)
(1989, 7)


,Date,News
0,2008-06-08,"Marriage, they said, was reduced to the status..."
1,2008-06-08,Nim Chimpsky: The tragedy of the chimp who tho...
2,2008-06-08,"Canada: Beware slippery slope' to censorship, ..."
3,2008-06-08,EU Vice-President Luisa Morgantini and the Iri...
4,2008-06-08,Israeli minister: Israel will attack Iran if i...


Headlines are aggregated by date so that each trading day corresponds to a single observation.

In [29]:
daily_headlines_df = aggregate_headlines_by_day(headlines_df)

daily_headlines_df.head()

,Date,text,headline_count
0,2008-06-08,"Marriage, they said, was reduced to the status...",25
1,2008-06-09,"Chew Qat: In Yemen, 72 per cent of men and 32 ...",25
2,2008-06-10,31 year old beats 3 year old to death: worst c...,25
3,2008-06-11,Pakistan Blames U.S. Coalition for Troops' Dea...,25
4,2008-06-12,"Marine ""Mean Motari"" gets expelled, another pu...",25


Additional features are computed from the market data, including daily return, intraday range, and changes relative to the previous day.

In [30]:
prices_df = add_price_features(prices_df)

prices_df.head()

,Date,Open,High,Low,Close,Volume,Adj Close,daily_return,intraday_range,close_vs_prev_close,volume_change
0,2008-08-08,11432.089844,11759.959961,11388.040039,11734.320312,212830000,11734.320312,0.026437,0.032533,NaN,NaN
1,2008-08-11,11729.669922,11867.110352,11675.530273,11782.349609,183190000,11782.349609,0.004491,0.016333,0.004093,-0.139266
2,2008-08-12,11781.700195,11782.349609,11601.519531,11642.469727,173590000,11642.469727,-0.011818,0.015348,-0.011872,-0.052405
3,2008-08-13,11632.809570,11633.780273,11453.339844,11532.959961,182550000,11532.959961,-0.008583,0.015511,-0.009406,0.051616
4,2008-08-14,11532.070312,11718.280273,11450.889648,11615.929688,159790000,11615.929688,0.007272,0.023187,0.007194,-0.124678


The headline and market datasets are merged on date, retaining only days where both sources are available.

In [31]:
dataset_df = merge_market_and_headlines(prices_df, daily_headlines_df)

dataset_df.head()

,Date,Open,High,Low,Close,Volume,Adj Close,daily_return,intraday_range,close_vs_prev_close,volume_change,text,headline_count
0,2008-08-08,11432.089844,11759.959961,11388.040039,11734.320312,212830000,11734.320312,0.026437,0.032533,NaN,NaN,Georgian troops retreat from S. Osettain capit...,25
1,2008-08-11,11729.669922,11867.110352,11675.530273,11782.349609,183190000,11782.349609,0.004491,0.016333,0.004093,-0.139266,Russia angered by Israeli military sale to Geo...,25
2,2008-08-12,11781.700195,11782.349609,11601.519531,11642.469727,173590000,11642.469727,-0.011818,0.015348,-0.011872,-0.052405,U.S. Beats War Drum as Iran Dumps the Dollar [...,25
3,2008-08-13,11632.809570,11633.780273,11453.339844,11532.959961,182550000,11532.959961,-0.008583,0.015511,-0.009406,0.051616,Bush announces Operation Get All Up In Russia'...,25
4,2008-08-14,11532.070312,11718.280273,11450.889648,11615.929688,159790000,11615.929688,0.007272,0.023187,0.007194,-0.124678,Poland and US agree to missle defense deal. In...,25


A binary target is defined based on next-day price movement. A value of 1 indicates that the next trading day closes higher than the current day, while a value of 0 indicates that it does not. This means flat and negative next-day movements are grouped together as the non-positive class.

In [32]:
dataset_df = add_next_day_target(dataset_df)

dataset_df.head()

,Date,Open,High,Low,Close,Volume,Adj Close,daily_return,intraday_range,close_vs_prev_close,volume_change,text,headline_count,next_close,next_day_return,target
0,2008-08-08,11432.089844,11759.959961,11388.040039,11734.320312,212830000,11734.320312,0.026437,0.032533,NaN,NaN,Georgian troops retreat from S. Osettain capit...,25,11782.349609,0.004093,1
1,2008-08-11,11729.669922,11867.110352,11675.530273,11782.349609,183190000,11782.349609,0.004491,0.016333,0.004093,-0.139266,Russia angered by Israeli military sale to Geo...,25,11642.469727,-0.011872,0
2,2008-08-12,11781.700195,11782.349609,11601.519531,11642.469727,173590000,11642.469727,-0.011818,0.015348,-0.011872,-0.052405,U.S. Beats War Drum as Iran Dumps the Dollar [...,25,11532.959961,-0.009406,0
3,2008-08-13,11632.809570,11633.780273,11453.339844,11532.959961,182550000,11532.959961,-0.008583,0.015511,-0.009406,0.051616,Bush announces Operation Get All Up In Russia'...,25,11615.929688,0.007194,1
4,2008-08-14,11532.070312,11718.280273,11450.889648,11615.929688,159790000,11615.929688,0.007272,0.023187,0.007194,-0.124678,Poland and US agree to missle defense deal. In...,25,11659.900391,0.003785,1


The dataset is split chronologically into training, validation, and test sets using a 70/15/15 ratio. This preserves temporal order while reserving later observations for model selection and final evaluation.

In [33]:
train_df, val_df, test_df = split_by_ratio(
    dataset_df,
    train_ratio=0.70,
    val_ratio=0.15,
)

print(len(train_df), len(val_df), len(test_df))

print("\nTrain range:", train_df["Date"].min(), "→", train_df["Date"].max())
print("Val range:", val_df["Date"].min(), "→", val_df["Date"].max())
print("Test range:", test_df["Date"].min(), "→", test_df["Date"].max())

1391 298 299

Train range: 2008-08-08 00:00:00 → 2014-02-18 00:00:00
Val range: 2014-02-19 00:00:00 → 2015-04-24 00:00:00
Test range: 2015-04-27 00:00:00 → 2016-06-30 00:00:00


## Phase 1: Predicting Market Direction from Headlines

This phase evaluates whether daily news headlines alone contain predictive signal for next-day market direction. Only headline text and the target variable are retained for this phase.

In [34]:
train_text_df = train_df[["text", "target"]].copy()
val_text_df = val_df[["text", "target"]].copy()
test_text_df = test_df[["text", "target"]].copy()

In [35]:
X_train = train_text_df["text"].astype(str).values
X_val = val_text_df["text"].astype(str).values
X_test = test_text_df["text"].astype(str).values

y_train = train_text_df["target"].values
y_val = val_text_df["target"].values
y_test = test_text_df["target"].values

### Baseline

A majority-class baseline is used as a reference point for model performance.

In [36]:
from sklearn.metrics import accuracy_score

majority_class = train_text_df["target"].mode()[0]

val_pred = [majority_class] * len(val_text_df)
test_pred = [majority_class] * len(test_text_df)

print("Majority class:", majority_class)
print("Validation accuracy:", accuracy_score(val_text_df["target"], val_pred))
print("Test accuracy:", accuracy_score(test_text_df["target"], test_pred))

Majority class: 1
Validation accuracy: 0.5536912751677853
Test accuracy: 0.5083612040133779


### Prepare inputs

In [37]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

X_train = train_text_df["text"].astype(str).values
X_val = val_text_df["text"].astype(str).values
X_test = test_text_df["text"].astype(str).values

y_train = train_text_df["target"].values
y_val = val_text_df["target"].values
y_test = test_text_df["target"].values

max_tokens = 10000
sequence_length = 200

vectorizer = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length,
)

vectorizer.adapt(X_train)

The data is converted into batched TensorFlow datasets for training.

In [38]:
batch_size = 32

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size)

An LSTM model is used to capture sequential patterns in the headline text.

In [39]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional


def create_lstm_model(
    vectorizer,
    max_tokens,
    embedding_dim=64,
    lstm_units=64,
    bidirectional=False,
    dense_units=None,
    dropout_rate=0.0,
):
    model = Sequential()
    model.add(vectorizer)
    model.add(Embedding(input_dim=max_tokens, output_dim=embedding_dim, mask_zero=True))

    lstm_layer = LSTM(
        lstm_units,
        dropout=dropout_rate,
        recurrent_dropout=dropout_rate,
    )

    if bidirectional:
        model.add(Bidirectional(lstm_layer))
    else:
        model.add(lstm_layer)

    if dense_units is not None:
        model.add(Dense(dense_units, activation="relu"))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model

Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 11s 169ms/step - accuracy: 0.5032 - loss: 0.6925 - val_accuracy: 0.5537 - val_loss: 0.6883
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 6s 129ms/step - accuracy: 0.6075 - loss: 0.6476 - val_accuracy: 0.5369 - val_loss: 0.8638
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 6s 148ms/step - accuracy: 0.5298 - loss: 1.1848 - val_accuracy: 0.4899 - val_loss: 0.6940
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 6s 135ms/step - accuracy: 0.6981 - loss: 0.6424 - val_accuracy: 0.5268 - val_loss: 0.6907


In [40]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print("Validation accuracy:", val_acc)
print("Majority baseline:", accuracy_score(val_text_df["target"], [majority_class] * len(val_text_df)))

Validation accuracy: 0.5536912679672241
Majority baseline: 0.5536912751677853


In [41]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

val_probs = model.predict(val_ds).ravel()
val_preds = (val_probs >= 0.5).astype(int)

print(classification_report(y_val, val_preds, digits=4))
print(confusion_matrix(y_val, val_preds))
print("Mean predicted probability:", val_probs.mean())

10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       133
           1     0.5537    1.0000    0.7127       165

    accuracy                         0.5537       298
   macro avg     0.2768    0.5000    0.3564       298
weighted avg     0.3066    0.5537    0.3946       298

[[  0 133]
 [  0 165]]
Mean predicted probability: 0.53108126


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
